# Evaluator Pattern — Build, Review, and Revise

This notebook demonstrates the evaluator MAS pattern: a `build_agent`
produces a result, a `review_agent` checks it against a checklist, and
the coordinator loops build to review to revise until the checklist
passes.

`review_agent`'s checklist is a deterministic tool, not an opinion.
`check_copy` runs real string checks (word count, an exact required
phrase, a list of banned words) and returns a structured report. The
coordinator only tells `build_agent` a short, high-level brief, not
the full checklist, so a first draft genuinely can miss something it
was never told, the same way a real style guide a writer doesn't have
memorized would.

In [1]:
# Uncomment the line below to install `llm-agents-from-scratch` from PyPI
# !pip install llm-agents-from-scratch

## Running an Ollama service

To execute the code provided in this notebook, you'll need to have
Ollama installed on your local machine and have its LLM hosting
service running. To download Ollama, follow the instructions found on
this page: https://ollama.com/download. After downloading and
installing Ollama, you can start a service by opening a terminal and
running `ollama serve`.

In [2]:
import os
import shutil
import subprocess
import time
import urllib.error
import urllib.request


def ensure_ollama(host="http://localhost:11434", timeout=15):
    """Start Ollama if not already running and wait until responsive."""

    def _up():
        try:
            urllib.request.urlopen(f"{host}/api/tags", timeout=1)
            return True
        except (urllib.error.URLError, ConnectionError, TimeoutError):
            return False

    if _up():
        return print(f"\u2713 Ollama already running at {host}")

    ollama_path = shutil.which("ollama")
    if ollama_path is None:
        for candidate in [
            "/teamspace/studios/this_studio/.local/bin/ollama",
            "/usr/local/bin/ollama",
            "/usr/bin/ollama",
        ]:
            if os.path.exists(candidate):
                ollama_path = candidate
                break
    if ollama_path is None:
        raise RuntimeError(
            "Could not find the ollama binary. Install with: "
            "curl -fsSL https://ollama.com/install.sh | sh",
        )

    print(f"Starting Ollama server ({ollama_path})...")
    subprocess.Popen(
        [ollama_path, "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

    deadline = time.time() + timeout
    while time.time() < deadline:
        if _up():
            return print(f"\u2713 Ollama up and running at {host}")
        time.sleep(0.5)

    raise RuntimeError(f"Ollama did not start within {timeout}s")


use_cloud = "OLLAMA_API_KEY" in os.environ
ensure_ollama() if not use_cloud else print("\u2713 Using Ollama Cloud")

✓ Using Ollama Cloud


In [3]:
model = "qwen3.5:397b-cloud" if use_cloud else "qwen3:14b"
host = "https://ollama.com" if use_cloud else None

## Defining the Checklist

`check_copy` is the house compliance checklist for a product
announcement: an exact word count range, a required phrase, a list of
banned marketing words, and an exact required call-to-action string.
None of this is negotiable or a matter of taste; it's the same kind
of style guide a real marketing or legal team might enforce.

In [4]:
import re

from llm_agents_from_scratch.tools.simple_function import SimpleFunctionTool

BANNED = ["best", "guaranteed", "revolutionary", "amazing", "incredible"]
REQUIRED_PHRASE = "offline mode"
CTA = "Learn more at ourproduct.com."
MIN_WORDS, MAX_WORDS = 30, 45


def check_copy(text: str) -> dict:
    """Check announcement copy against the house compliance checklist."""
    issues = []
    word_count = len(text.split())
    if not (MIN_WORDS <= word_count <= MAX_WORDS):
        issues.append(
            f"word count {word_count} outside {MIN_WORDS}-{MAX_WORDS}",
        )
    if REQUIRED_PHRASE.lower() not in text.lower():
        issues.append(f"missing required phrase {REQUIRED_PHRASE!r}")
    found = [w for w in BANNED if re.search(rf"\b{w}\b", text, re.IGNORECASE)]
    if found:
        issues.append(f"uses banned word(s): {', '.join(found)}")
    if not text.strip().endswith(CTA):
        issues.append(f"must end with the exact call-to-action: {CTA!r}")
    return {"compliant": not issues, "word_count": word_count, "issues": issues}


check_copy_tool = SimpleFunctionTool(func=check_copy)

/home/nerdai/Projects/llm-agents-from-scratch/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Defining the Specialists

`build_agent` only ever sees the brief below, never the checklist
above. `review_agent` never sees the checklist either, in the sense
that it doesn't recite it; it just calls `check_copy` and reports
whatever the tool actually returns.

In [5]:
from llm_agents_from_scratch import LLMAgent, LLMAgentBuilder
from llm_agents_from_scratch.data_structures import Task
from llm_agents_from_scratch.llms import OllamaLLM
from llm_agents_from_scratch.subagents import SubAgentSpec

llm = OllamaLLM(host=host, model=model, think=False, json_prompt_mode=use_cloud)

build_agent = SubAgentSpec(
    name="build_agent",
    description="Writes marketing copy to a given brief.",
    builder=LLMAgentBuilder(llm=llm),
    max_steps=5,
)
review_agent = SubAgentSpec(
    name="review_agent",
    description="Checks copy against the house compliance checklist.",
    builder=LLMAgentBuilder(llm=llm, tools=[check_copy_tool]),
    max_steps=5,
)

## Example — Converging on a Compliant Draft

The brief below gives `build_agent` the word count, the required
phrase, and asks for a call to action, but not its exact wording,
and nothing about banned words. Both are real requirements
`check_copy` enforces regardless.

In [6]:
brief = (
    f"Write a {MIN_WORDS}-{MAX_WORDS} word product announcement for a "
    f"note-taking app's new {REQUIRED_PHRASE!r} feature, ending with a "
    "call to action to learn more. Return only the announcement text."
)

coordinator = LLMAgent(llm=llm, subagents=[build_agent, review_agent])

task = Task(
    instruction=(
        f"Ask build_agent to: {brief} "
        "Then ask review_agent to check the copy against the house "
        "compliance checklist and report any issues. If there are "
        "issues, ask build_agent to revise using the specific issues "
        "reported (quote them exactly), then ask review_agent to check "
        "again. Repeat until compliant, up to 3 rounds. Report the "
        "final copy and whether it's compliant."
    ),
)
handler = coordinator.run(task, max_steps=15)
result = await handler

In [7]:
print(result.content)

The review_agent has confirmed that the revised copy is now compliant. Let me summarize the final results:

**Final Copy:**
"Capture ideas anytime, anywhere—even without Wi-Fi. Our new offline mode ensures your notes sync automatically once you're back online. Never lose a thought again. Update now and experience seamless productivity. Learn more at ourproduct.com."

**Compliance Status:** Compliant (Yes)
- Word Count: 34 words (within the 30-45 word requirement)
- Issues: None found

The copy went through 1 revision round to fix the call-to-action issue, and it is now fully compliant with the house compliance checklist.

`handler.rollout` shows why a revision was needed: `review_agent`'s
tool result names the specific missing call-to-action, straight from
`check_copy`'s own return value, not a guess, and the coordinator
relays that specific issue back to `build_agent` rather than just
asking it to try again.

In [8]:
print(handler.rollout)

=== Task Step Start ===

💬 assistant: My current instruction is 'Ask build_agent to: Write a 30-45 word product announcement for a note-taking app's new 'offline mode' feature, ending with a call to action to learn more. Return only the announcement text. Then ask review_agent to check the copy against the house compliance checklist and report any issues. If there are issues, ask build_agent to revise using the specific issues reported (quote them exactly), then ask review_agent to check again. Repeat until compliant, up to 3 rounds. Report the final copy and whether it's compliant.'

💬 assistant: I need to make the following tool call(s):

{
    "id_": "811d46e4-67b3-43a7-bf27-71d08d749187",
    "tool_name": "from_scratch__use_subagent",
    "arguments": {
        "name": "build_agent",
        "task": "Write a 30-45 word product announcement for a note-taking app's new 'offline mode' feature, ending with a call to action to learn more. Return only the announcement text."
    }
}.

🔧 

The 3-round cap is a safety net, not something this example needs.
`review_agent` never invents requirements, so once `build_agent`
finally sees the specific issue spelled out, satisfying it on the
next attempt is straightforward.